# 04 — Evaluation Analysis

Load eval result JSON files and visualise retrieval quality and faithfulness metrics.

In [22]:
from pathlib import Path
import json

BASE_DIR = Path.cwd().parent  # move up from notebooks/
EVAL_DIR = BASE_DIR / 'data/processed/metadata'

files = sorted(EVAL_DIR.glob('eval_results_*.json'))

print(files)  # debug

if not files:
    print('No eval results found.')
else:
    latest = json.loads(files[-1].read_text())
    print(latest['strategy'])
    print('Questions:', latest['num_questions'])

[WindowsPath('c:/Users/dipes/Desktop/Projects/financial-rag-copilot/data/processed/metadata/eval_results_multi_query.json'), WindowsPath('c:/Users/dipes/Desktop/Projects/financial-rag-copilot/data/processed/metadata/eval_results_similarity.json')]
similarity
Questions: 6


In [26]:
import pandas as pd

summary = latest.get('summary', {})
rows = []
for category, metrics in summary.items():
    if isinstance(metrics, dict):
        rows.append({'category': category, **{k: v for k, v in metrics.items() if v is not None}})

pd.DataFrame(rows)

,category,count,retrieval_hit_rate,section_hit_rate,parent_hit_rate,mean_precision_at_k,mean_recall_at_k,mean_answer_relevance,mean_key_point_coverage,insufficient_context_compliance_rate,hallucination_rate,mean_supported_sentence_ratio,citation_presence_rate,unsupported_answer_rate
0,retrieval,6,0.833333,0.833333,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,answer,6,NaN,NaN,NaN,NaN,NaN,0.567827,0.0,0.833333,0.0,NaN,NaN,NaN
2,faithfulness,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.75,1.0,0.333333


In [27]:
# Per-question breakdown
df = pd.DataFrame(latest.get('details', []))
df.head()

,question_id,question,ticker,form_type,strategy,answer,citations,retrieved_chunks,retrieval_eval,answer_eval,faithfulness_eval,schema_valid
0,q001,What cybersecurity risks does Apple disclose i...,AAPL,10-K,similarity,"{\n ""answer"": ""Apple discloses several cybers...","[{'citation_id': 'C1', 'chunk_id': '7738bcede0...",[{'text': 'Although malicious attacks perpetra...,"{'question_id': 'q001', 'retrieved_chunk_ids':...","{'question_id': 'q001', 'answer_relevance': 0....","{'question_id': 'q001', 'supported_sentence_ra...",True
1,q002,What supply chain risks does Apple highlight i...,AAPL,10-K,similarity,"{\n ""answer"": ""In its latest 10-K, Apple high...","[{'citation_id': 'C1', 'chunk_id': '3559b971bc...","[{'text': 'Therefore, the Company remains subj...","{'question_id': 'q002', 'retrieved_chunk_ids':...","{'question_id': 'q002', 'answer_relevance': 0....","{'question_id': 'q002', 'supported_sentence_ra...",True
2,q003,What macroeconomic risks does Apple identify i...,AAPL,10-K,similarity,"{\n ""answer"": ""In its latest 10-K, Apple iden...","[{'citation_id': 'C1', 'chunk_id': '6d66244c61...",[{'text': 'Adverse economic conditions can als...,"{'question_id': 'q003', 'retrieved_chunk_ids':...","{'question_id': 'q003', 'answer_relevance': 0....","{'question_id': 'q003', 'supported_sentence_ra...",True
3,q004,What changes in financial reporting standards ...,AAPL,10-K,similarity,"{\n ""answer"": ""The provided context does not ...","[{'citation_id': 'C1', 'chunk_id': '307302436e...",[{'text': 'The Company’s Annual Reports on For...,"{'question_id': 'q004', 'retrieved_chunk_ids':...","{'question_id': 'q004', 'answer_relevance': 0....","{'question_id': 'q004', 'supported_sentence_ra...",True
4,q005,What does Apple say about adverse economic con...,AAPL,10-K,similarity,"{\n ""answer"": ""In its latest 10-K, Apple stat...","[{'citation_id': 'C1', 'chunk_id': '6d66244c61...",[{'text': 'Adverse economic conditions can als...,"{'question_id': 'q005', 'retrieved_chunk_ids':...","{'question_id': 'q005', 'answer_relevance': 0....","{'question_id': 'q005', 'supported_sentence_ra...",True


In [29]:
# Compare strategies across multiple runs
all_runs = [json.loads(Path(f).read_text()) for f in files]

strategy_rows = []
for run in all_runs:
    s = run.get('summary', {})
    
    strategy_rows.append({
        # fallback if run_at not present
        'run_at': run.get('run_at', 'unknown')[:16] if run.get('run_at') else 'unknown',
        
        'strategy': run.get('strategy', '?'),
        
        # ✅ correct keys
        'retrieval_hit_rate': s.get('retrieval', {}).get('retrieval_hit_rate'),
        'section_hit_rate': s.get('retrieval', {}).get('section_hit_rate'),
        
        'faithfulness': s.get('faithfulness', {}).get('mean_supported_sentence_ratio'),
        'unsupported_rate': s.get('faithfulness', {}).get('unsupported_answer_rate'),
        
        'answer_relevance': s.get('answer', {}).get('mean_answer_relevance'),
        
        'schema_validity': s.get('schema_validity_rate'),
    })

df = pd.DataFrame(strategy_rows)
df

,run_at,strategy,retrieval_hit_rate,section_hit_rate,faithfulness,unsupported_rate,answer_relevance,schema_validity
0,unknown,multi_query,0.833333,0.833333,0.75,0.333333,0.569511,1.0
1,unknown,similarity,0.833333,0.833333,0.75,0.333333,0.567827,1.0
